# Spec-Craft Quickstart: Spec-Driven Development with AI & Obsidian

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akuroiwa/spec-craft/blob/main/docs/notebooks/spec_craft_quickstart.ipynb)

Welcome to the **Spec-Craft** interactive tutorial on Google Colab!

**Spec-Craft** is an Obsidian-based Spec-Driven Development (SDD) toolkit designed for AI agents and developers. It establishes a clean separation of concerns:
- **Strategy (Obsidian)**: Defines the *What* and *Why* (the project's soul).
- **Tactics (spec-kit & code)**: Executes the *How* (iterative implementation cycles).

In this notebook, you will learn how to:
1. Set up a local LLM environment using **Ollama** and **sigoden/aichat** with `uv`.
2. Generate an Obsidian strategic feature specification (`obsidian/strategy_task_tracker.md`).
3. Inspect and parse the strategic roadmap using **Spec-Craft**'s `StoryboardParser`.
4. Automatically bridge strategy into tactical implementation (generating Python code and `pytest` test suites).
5. Verify behavioral correctness by running tests directly in Colab.

In [ ]:
# ==============================================================================
# 1. Environment Setup (uv, sigoden/aichat, Ollama, and spec-craft)
# ==============================================================================
print("📦 Installing system dependencies and uv...")
!apt-get update -qq && apt-get install -y -qq curl zstd git
!pip install -q uv

print("📥 Installing sigoden/aichat CLI...")
!curl -sSF https://raw.githubusercontent.com/sigoden/aichat/main/scripts/install.sh | sh -s -- --to /usr/local/bin

print("🚀 Installing and starting Ollama background service...")
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, os
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# --- Model Selection (Default: Fast & Stable for Colab T4/CPU) ---
print("🧠 Pulling LLM model (Qwen 2.5 1.5B)...")
!ollama pull qwen2.5:1.5b

# --- Alternative Hugging Face / QAT Models ---
# !ollama pull hf.co/unsloth/gemma-2-2b-it-GGUF
# !ollama pull gemma2:2b

print("🛠️ Installing spec-craft...")
!uv pip install --system git+https://github.com/akuroiwa/spec-craft.git

# Configure aichat to use Ollama
aichat_config = """
model: ollama:qwen2.5:1.5b
"""
os.makedirs(os.path.expanduser("~/.config/aichat"), exist_ok=True)
with open(os.path.expanduser("~/.config/aichat/config.yaml"), "w") as f:
    f.write(aichat_config.strip())

print("✅ Environment setup complete!")

## 1. Verify AI Chat & spec-craft Prerequisites

Check the environment readiness using `spec-craft check-env` and run a quick query via `aichat`.

In [ ]:
# Verify environment readiness
!spec-craft check-env

# Test aichat query
!aichat "Hello! Confirm in one short sentence that you are ready for Spec-Driven Development."

## 2. Step 1: Draft the Strategic Specification in Obsidian (`obsidian/`)

In Spec-Craft, development begins with a strategic vision in Obsidian. We define the **What** and **Why** before diving into implementation details.

Let's prompt `aichat` to draft a feature specification for a **Markdown Task Progress Analyzer CLI tool** (`obsidian/strategy_task_tracker.md`).

In [ ]:
import os
os.makedirs("obsidian", exist_ok=True)

# Draft Obsidian specification with frontmatter and structured sections
!aichat "Create an Obsidian-compatible feature roadmap Markdown note with YAML frontmatter. \
Title: 'Markdown Task Tracker CLI'. \
Include YAML frontmatter with title, tags: [sdd, cli, utility], and status: draft. \
Include H1 sections: '# 1. Overview', '# 2. User Stories', '# 3. Technical Requirements'. \
The tool parses a markdown string or file, finds all '- [ ]' (pending) and '- [x]' (completed) checklist items, and calculates total, completed, pending, and completion percentage." > obsidian/strategy_task_tracker.md

# Display the generated Obsidian strategic specification
!cat obsidian/strategy_task_tracker.md

## 3. Step 2: Inspect and Parse the Strategic Roadmap with Spec-Craft

Spec-Craft provides structured parsers and MCP tools to inspect Obsidian storyboards, extracting metadata and sections safely without losing context.

In [ ]:
from pathlib import Path
from spec_craft.core.parser import StoryboardParser

# Parse the generated storyboard note
parser = StoryboardParser(Path("obsidian"))
storyboard = parser.parse("strategy_task_tracker.md")

print(f"📄 File Path: {storyboard.file_path}")
print(f"🏷️  Metadata: {storyboard.metadata}")
print(f"📑 Sections Found ({len(storyboard.sections)}):")
for section in storyboard.sections:
    print(f"   [Level {section.level}] {section.title}")

## 4. Step 3: Automated Tactical Implementation (spec-kit Workflow)

Following Spec-Craft's SDD lifecycle:
1. `spec-craft` guides the tactical phase (translating Strategy into executable Python code and tests).
2. We generate the implementation module (`task_tracker.py`) and unit tests (`test_task_tracker.py`).

In [ ]:
# Generate the Python implementation based on the Obsidian spec
!aichat -f obsidian/strategy_task_tracker.md "Implement the complete Python module 'task_tracker.py'. \
Provide a class 'TaskTracker' and a function 'analyze_tasks(content: str) -> dict' \
returning {'total': int, 'completed': int, 'pending': int, 'percentage': float}. \
Output ONLY raw Python code without Markdown backticks." > task_tracker.py

# Generate unit tests with pytest
!aichat -f task_tracker.py "Write comprehensive pytest unit tests in 'test_task_tracker.py'. \
Test empty text, all pending, all completed, and mixed tasks. Output ONLY raw Python code without Markdown backticks." > test_task_tracker.py

print("✨ Generated task_tracker.py and test_task_tracker.py")

## 5. Step 4: Verification and Test Execution

Run `pytest` to verify that the generated implementation passes all acceptance criteria.

In [ ]:
# Execute unit tests
!pytest test_task_tracker.py -v

## Summary & Next Steps

Congratulations! You have completed the full Spec-Driven Development cycle in Google Colab:
- **Strategy (Obsidian)**: Captured vision and user requirements.
- **Spec-Craft & aichat**: Parsed structure and bridged strategy to tactics.
- **Tactics (spec-kit & pytest)**: Generated clean, test-verified implementation.

### Resources
- **GitHub Repository**: [https://github.com/akuroiwa/spec-craft](https://github.com/akuroiwa/spec-craft)
- **Documentation**: [https://spec-craft.readthedocs.io/](https://spec-craft.readthedocs.io/)
- **Zenn Book / Articles**: [spec-craft Development Notes](https://zenn.dev/akuroiwa/books/dev-note-spec-craft)